# 14 GLM M1: Structural Expected-Citation Model

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

GENDER_CATEGORIES = ["MM", "MW", "WM", "WW"]

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BC_DATA_DIR = PROJECT_ROOT / "outputs" / "04_bibliographic_coupling_network" / "data"
COCIT_DATA_DIR = PROJECT_ROOT / "outputs" / "03_cocitation_network" / "data"

NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "14_baseline_m1_combined_glm"
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / "figures"
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / "tables"
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "data"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Functions

In [2]:
def standardize_edges(edges):
    out = edges[["citing_paper_id", "cited_paper_id"]].copy()
    out["citing_paper_id"] = out["citing_paper_id"].astype(str)
    out["cited_paper_id"] = out["cited_paper_id"].astype(str)
    return out

def load_final_core():
    papers = pd.read_pickle(PROCESSED_DIR / "papers_final.pkl")
    cs_papers = pd.read_pickle(PROCESSED_DIR / "cs_papers_final.pkl")
    all_edges = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_final.pkl"))
    all_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_all_to_cs_final.pkl"))
    cs_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_cs_to_cs_final.pkl"))
    papers["id"] = papers["id"].astype(str)
    cs_papers["id"] = cs_papers["id"].astype(str)
    return papers, cs_papers, all_edges, all_to_cs, cs_to_cs

def add_observed(papers_df, edges, col="observed"):
    counts = edges["cited_paper_id"].value_counts()
    out = papers_df.copy()
    out[col] = out["paper_id"].map(counts).fillna(0).astype(float)
    return out

def make_structural_base(cs_papers):
    x = cs_papers[cs_papers["first_last_cat"].isin(GENDER_CATEGORIES)].copy()
    x = x.rename(columns={"id": "paper_id", "first_last_cat": "gender_cat"})
    x["paper_id"] = x["paper_id"].astype(str)
    x["log1p_n_refs"] = np.log1p(pd.to_numeric(x["n_refs"], errors="coerce").fillna(0).clip(lower=0))
    x["decade"] = x["decade"].astype("string").fillna("Unknown")
    x["primary_subfield"] = x["primary_subfield"].astype("string").fillna("Unknown")
    x["venue_rank_clean"] = x["venue_rank_clean"].astype("string").fillna("N/A")
    return x

def fit_poisson_expected(df, outcome_col, categorical_features, numeric_features, expected_col, alpha=1.0):
    x = df.copy()
    for c in categorical_features:
        x[c] = x[c].astype("string").fillna("Unknown")
    for c in numeric_features:
        x[c] = pd.to_numeric(x[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X = x[categorical_features + numeric_features]
    y = x[outcome_col].astype(float).to_numpy()
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ])
    pipe = Pipeline([
        ("preprocess", pre),
        ("model", PoissonRegressor(alpha=alpha, max_iter=300, tol=1e-6)),
    ])
    pipe.fit(X, y)
    raw = np.clip(pipe.predict(X), 1e-12, None)
    scale = y.sum() / raw.sum()
    x[expected_col] = raw * scale
    return x, pipe, scale

def aggregate_glm_oe(df, outcome_col, expected_col, model_name, citation_slice):
    out = (
        df.groupby("gender_cat")
        .agg(
            n_papers=("paper_id", "nunique"),
            observed_citations=(outcome_col, "sum"),
            expected_citations=(expected_col, "sum"),
        )
        .reindex(GENDER_CATEGORIES)
        .reset_index()
    )
    out["oe_ratio"] = out["observed_citations"] / out["expected_citations"].replace(0, np.nan)
    out["pct_diff_from_expected"] = (out["oe_ratio"] - 1) * 100
    out["model"] = model_name
    out["citation_slice"] = citation_slice
    return out

def standardize_bc(net):
    out = net[["paper_i", "paper_j", "shared_references"]].rename(
        columns={"shared_references": "weight"}
    ).copy()
    out["paper_i"] = out["paper_i"].astype(str)
    out["paper_j"] = out["paper_j"].astype(str)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")
    return out[out["weight"].gt(0) & out["paper_i"].ne(out["paper_j"])].drop_duplicates(["paper_i", "paper_j"])

def standardize_cocit(net):
    out = net[["paper_a", "paper_b", "shared_citers"]].rename(
        columns={"paper_a": "paper_i", "paper_b": "paper_j", "shared_citers": "weight"}
    ).copy()
    out["paper_i"] = out["paper_i"].astype(str)
    out["paper_j"] = out["paper_j"].astype(str)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")
    return out[out["weight"].gt(0) & out["paper_i"].ne(out["paper_j"])].drop_duplicates(["paper_i", "paper_j"])

def direct_cs_features(cs_to_cs):
    g = cs_to_cs.groupby("citing_paper_id")
    out = pd.DataFrame({
        "paper_id": g.size().index,
        "direct_out_degree": g["cited_paper_id"].nunique().values,
    })
    out["log1p_direct_out_degree"] = np.log1p(out["direct_out_degree"])
    return out[["paper_id", "log1p_direct_out_degree"]]

def network_node_features(net, prefix):
    fwd = net.rename(columns={"paper_i": "paper_id", "paper_j": "neighbor_id"})
    rev = net.rename(columns={"paper_j": "paper_id", "paper_i": "neighbor_id"})
    d = pd.concat([fwd, rev], ignore_index=True)
    g = (
        d.groupby("paper_id", sort=False)
        .agg(
            degree=("neighbor_id", "nunique"),
            strength=("weight", "sum"),
            mean_weight=("weight", "mean"),
            max_weight=("weight", "max"),
        )
        .reset_index()
    )
    for c in ["degree", "strength", "mean_weight", "max_weight"]:
        g[f"log1p_{prefix}_{c}"] = np.log1p(g[c].clip(lower=0))
    return g[[
        "paper_id",
        f"log1p_{prefix}_degree",
        f"log1p_{prefix}_strength",
        f"log1p_{prefix}_mean_weight",
        f"log1p_{prefix}_max_weight",
    ]]

def load_network_features(base, edges_cs_to_cs):
    x = base.merge(direct_cs_features(edges_cs_to_cs), on="paper_id", how="left")
    bib = standardize_bc(pd.read_pickle(BC_DATA_DIR / "net_bib_coupling_topk.pkl"))
    cocit = standardize_cocit(pd.read_feather(COCIT_DATA_DIR / "cocitation_edges.feather"))
    x = x.merge(network_node_features(bib, "bib"), on="paper_id", how="left")
    x = x.merge(network_node_features(cocit, "cocit"), on="paper_id", how="left")
    network_cols = [
        c for c in x.columns
        if c.startswith("log1p_direct_") or c.startswith("log1p_bib_") or c.startswith("log1p_cocit_")
    ]
    x[network_cols] = x[network_cols].fillna(0.0)
    return x, bib, cocit

## Fit M1

In [3]:
papers, cs_papers, citation_edges_final, edges_all_to_cs, edges_cs_to_cs = load_final_core()
base = make_structural_base(cs_papers)
base_all = add_observed(base, edges_all_to_cs, "observed")
base_cs = add_observed(base, edges_cs_to_cs, "observed")

CATEGORICAL = ["decade", "primary_subfield", "venue_rank_clean"]
NUMERIC = ["log1p_n_refs"]

fit_all, pipe_all, scale_all = fit_poisson_expected(
    base_all, "observed", CATEGORICAL, NUMERIC, "expected", alpha=1.0
)
fit_cs, pipe_cs, scale_cs = fit_poisson_expected(
    base_cs, "observed", CATEGORICAL, NUMERIC, "expected", alpha=1.0
)

oe = pd.concat([
    aggregate_glm_oe(fit_all, "observed", "expected", "M1_structural", "All→CS"),
    aggregate_glm_oe(fit_cs, "observed", "expected", "M1_structural", "CS→CS"),
], ignore_index=True)

display(oe)

,gender_cat,n_papers,observed_citations,expected_citations,oe_ratio,pct_diff_from_expected,model,citation_slice
0,MM,790617,"6,942,653.0000","6,513,816.3175",1.0658,6.5835,M1_structural,All→CS
1,MW,97399,"641,435.0000","743,870.8941",0.8623,-13.7707,M1_structural,All→CS
2,WM,137756,"812,529.0000","1,032,371.2810",0.7871,-21.2949,M1_structural,All→CS
3,WW,55591,"289,620.0000","396,178.5074",0.7310,-26.8966,M1_structural,All→CS
4,MM,790617,"5,601,920.0000","5,252,988.7420",1.0664,6.6425,M1_structural,CS→CS
5,MW,97399,"529,022.0000","605,704.8547",0.8734,-12.6601,M1_structural,CS→CS
6,WM,137756,"662,200.0000","842,557.0541",0.7859,-21.4059,M1_structural,CS→CS
7,WW,55591,"230,358.0000","322,249.3493",0.7148,-28.5156,M1_structural,CS→CS


## Save Results

In [4]:
oe.to_csv(TABLE_DIR / "m1_structural_glm_oe_by_gender.csv", index=False)
fit_all[["paper_id", "gender_cat", "observed", "expected"]].to_pickle(
    DATA_OUTPUT_DIR / "m1_all_to_cs_paper_predictions.pkl"
)
fit_cs[["paper_id", "gender_cat", "observed", "expected"]].to_pickle(
    DATA_OUTPUT_DIR / "m1_cs_to_cs_paper_predictions.pkl"
)